In [1]:
from bert_classfifer import BertClassifier

/Users/tylersoong/projects/math373/naivebayes/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Getting texts and labels

In [2]:
from email.parser import BytesParser
from email import policy
import os
from tqdm import tqdm
from collections import Counter
import pandas as pd


email_path = "TRAINING/"

labels = {}
with open('SPAMTrain.label', 'r') as f:
    for line in f.readlines():
        splitted = line.split()
        labels[splitted[1]] = splitted[0]

rows = []
for file_name in tqdm(os.listdir(email_path)):
    label = labels[file_name]
    try:
        with open(os.path.join(email_path, file_name), 'rb') as fp:
            msg = BytesParser(policy=policy.default).parse(fp)

        body = msg.get_body(preferencelist=('plain'))
        if body:
            rows.append((body.get_content(), labels[file_name]))
    except Exception as e:
        print(f"Skipping {file_name}: {e}")


df = pd.DataFrame(rows, columns=['text', 'labels'])

100%|██████████| 4327/4327 [00:01<00:00, 2319.76it/s]

Skipping TRAIN_04057.eml: unknown encoding: DEFAULT


In [3]:
transformer_classifier = BertClassifier()
transformer_classifier.fit(
    texts=list(df['text'][:3000]),
    y=list(df['labels'][:3000].astype(int))  # ← force to plain Python list
)

`torch_dtype` is deprecated! Use `dtype` instead!
W0317 07:39:25.373000 7850 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1258.22it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/User

: 

In [ ]:
transformer_classifier.get_embeddings(list(df['text'][:10]))
vectors = pd.DataFrame(transformer_classifier.embeddings)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
to_graph = pca.fit_transform(vectors)



In [ ]:
print(df['labels'][:10])

0    1
1    1
2    1
3    1
4    1
5    1
6    0
7    1
8    1
9    1
Name: labels, dtype: str
